# Plan B — Fine-tune RAD-DINO trên CheXpert (14-label) + trích embedding 768-dim

**Luồng:** cohort = CXR ∩ EHR(ICD-10) → cache ảnh 256px một lần (resumable) → fine-tune RAD-DINO *last-N blocks* trên 14 nhãn CheXpert (U-Ones cho -1) → đánh giá 14-label (AUROC + precision/recall/F1) → trích embedding **768-dim** (CLS + mean-pool) từ backbone đã fine-tune.

**Chạy theo thứ tự.** Sau khi restart runtime, chạy lại từ cell config. Cache nằm trên Drive nên không mất; train đọc từ bản copy local cho nhanh.

> Cần đã có `planA_out/manifest.parquet` (từ notebook Plan A). RAD-DINO có thể cần đăng nhập HuggingFace.

## 1. Môi trường

In [ ]:
!pip install -q transformers timm huggingface_hub pyarrow scikit-image seaborn

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

# Nếu RAD-DINO yêu cầu token, bỏ comment và đăng nhập (KHÔNG hardcode token):
from huggingface_hub import login; login()

torch 2.11.0+cpu | transformers 5.12.1
cuda: False


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

Mounted at /content/drive


In [ ]:
from google.cloud import bigquery
import pandas as pd, glob, os
BILLING_PROJECT = 'mimic-ehr-v310'
client = bigquery.Client(project=BILLING_PROJECT)
H = "physionet-data.mimiciv_3_1_hosp"

# bệnh nhân từng có chẩn đoán HIV (ICD-9 + ICD-10)
q = f"""
SELECT DISTINCT subject_id FROM `{H}.diagnoses_icd`
WHERE (icd_version=10 AND (icd_code LIKE 'B20%' OR icd_code LIKE 'Z21%'
        OR icd_code LIKE 'B9735%' OR icd_code LIKE 'O987%'))
   OR (icd_version=9  AND (icd_code LIKE '042%' OR icd_code LIKE 'V08%'
        OR icd_code LIKE '07953%'))
"""
hiv_ids = set(client.query(q).to_dataframe().subject_id)
print("bệnh nhân HIV (ICD):", len(hiv_ids))

DIRS = ['/content/drive/MyDrive/MIMIC_CXR/planB_out',
        '/content/drive/MyDrive/MIMIC_CXR/planB_firstobs_out']

def add_hiv(d):
    if 'hiv' in d.columns: d = d.drop(columns='hiv')        # chạy lại không nhân đôi (chỉ hiv)
    d['hiv'] = d['subject_id'].isin(hiv_ids).astype('int8')
    return d

for D in DIRS:
    # 1) PARQUET (nguồn chuẩn, giữ đủ ảnh)
    for p in glob.glob(os.path.join(D,'fusion_dataset_*.parquet')):
        d = add_hiv(pd.read_parquet(p)); d.to_parquet(p, index=False)
        print("parquet:", os.path.basename(p), "| hiv dương:", int(d.hiv.sum()),
              "| img:", sum(c.startswith('img_') for c in d.columns), "| cột:", d.shape[1])
    # 2) CSV (giữ đúng dạng: full nếu vốn có ảnh, light nếu không)
    for cp in glob.glob(os.path.join(D,'fusion_dataset_*.csv')):
        had_img = any(c.startswith('img_') for c in pd.read_csv(cp, nrows=0).columns)
        d = add_hiv(pd.read_csv(cp))
        if not had_img: d = d[[c for c in d.columns if not c.startswith('img_')]]
        d.to_csv(cp, index=False)
        print("csv:", os.path.basename(cp), "| hiv dương:", int(d.hiv.sum()), "| img giữ:", had_img)


bệnh nhân HIV (ICD): 2757
parquet: fusion_dataset_hybrid.parquet | hiv dương: 6269 | img: 768 | cột: 861
parquet: fusion_dataset_hybrid_fullmicro.parquet | hiv dương: 6269 | img: 768 | cột: 1538
csv: fusion_dataset_hybrid.csv | hiv dương: 6269 | img giữ: False
csv: fusion_dataset_hybrid_full.csv | hiv dương: 6269 | img giữ: True
csv: fusion_dataset_hybrid_fullmicro.csv | hiv dương: 6269 | img giữ: True
parquet: fusion_dataset_encounter.parquet | hiv dương: 1002 | img: 768 | cột: 861
parquet: fusion_dataset_firstadm.parquet | hiv dương: 1002 | img: 768 | cột: 861
parquet: fusion_dataset_hybrid.parquet | hiv dương: 1160 | img: 768 | cột: 861
parquet: fusion_dataset_hybrid_fullmicro.parquet | hiv dương: 1160 | img: 768 | cột: 1538
csv: fusion_dataset_encounter.csv | hiv dương: 1002 | img giữ: True
csv: fusion_dataset_firstadm.csv | hiv dương: 1002 | img giữ: True
csv: fusion_dataset_hybrid.csv | hiv dương: 1160 | img giữ: False
csv: fusion_dataset_hybrid_full.csv | hiv dương: 1160 | img g

## 2. Mount + cấu hình

In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

import os
BASE        = '/content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org'
PLANA_DIR   = '/content/drive/MyDrive/MIMIC_CXR/planA_out'
OUT_DIR     = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
CACHE_DIR   = os.path.join(OUT_DIR, 'cache')     # cache uint8 256 trên Drive (bền)
LOCAL_CACHE = '/content/planB_cache'             # copy về local để train nhanh
for d in (OUT_DIR, CACHE_DIR, LOCAL_CACHE): os.makedirs(d, exist_ok=True)

BILLING_PROJECT = 'mimic-ehr-v310'
REPO            = 'microsoft/rad-dino'

CACHE_RES   = 256
TRAIN_RES   = 224
CACHE_SHARD = 4000
INCLUDE_LATERAL_TRAIN = True     # train/eval trên MỌI view (frontal+lateral+other)

N_UNFREEZE  = 3
EPOCHS      = 12
BATCH       = 64
LR_BACKBONE = 1e-5
LR_HEAD     = 1e-3
WEIGHT_DECAY= 0.05

CHEXPERT_LABELS = [
 'Atelectasis','Cardiomegaly','Consolidation','Edema','Enlarged Cardiomediastinum',
 'Fracture','Lung Lesion','Lung Opacity','No Finding','Pleural Effusion',
 'Pleural Other','Pneumonia','Pneumothorax','Support Devices']
Y_COLS = ['y_'+c for c in CHEXPERT_LABELS]

# ==== Cohort theo TÊN CHẨN ĐOÁN (long_title) — gồm cả ICD-9 + ICD-10, lấy TỐI ĐA ca ====
H = "physionet-data.mimiciv_3_1_hosp"
I = "physionet-data.mimiciv_3_1_icu"
DISEASE_TITLE = {
  'pneumonia':      "{t} LIKE '%pneumonia%'",
  'bronchitis':     "{t} LIKE '%bronchitis%'",
  'tuberculosis':   "{t} LIKE '%tuberculosis%'",
  'arthritis':      "{t} LIKE '%arthritis%'",
  'fracture':       "{t} LIKE '%fracture%'",
  'scoliosis':      "{t} LIKE '%scoliosis%'",
  'lung_cancer':    "({t} LIKE '%malignant%' AND {t} LIKE '%lung%')",
  'lung_infection': "(({t} LIKE '%lung%' OR {t} LIKE '%pulmonary%' OR {t} LIKE '%respiratory%') AND {t} LIKE '%infection%')",
}
DISEASE_CODES = {
  'pneumonia':      {'icd10':['J12','J13','J14','J15','J16','J17','J18','J69'], 'icd9':['480','481','482','483','484','485','486','5070']},
  'bronchitis':     {'icd10':['J20','J21','J40','J41','J42'], 'icd9':['466','490','491']},
  'tuberculosis':   {'icd10':['A15','A16','A17','A18','A19'], 'icd9':['010','011','012','013','014','015','016','017','018']},
  'arthritis':      {'icd10':['M00','M01','M02','M05','M06','M08','M13','M15','M16','M17','M18','M19'], 'icd9':['711','712','713','714','715','716']},
  'fracture':       {'icd10':['S02','S12','S22','S32','S42','S52','S62','S72','S82','S92','T02','T08','T10','T12','M80'], 'icd9':['80','81','82','7331']},
  'scoliosis':      {'icd10':['M41','Q675','Q763'], 'icd9':['7373','7542']},
  'lung_cancer':    {'icd10':['C34','C780'], 'icd9':['162','1970']},
  'lung_infection': {'icd10':['J22','J85','J86','B44','B59','B371'], 'icd9':['510','513','1173','1363','1124']},
}
def _code(spec):
    p=[]
    if spec.get('icd10'): p.append("(d.icd_version=10 AND ("+" OR ".join(f"d.icd_code LIKE '{c}%'" for c in spec['icd10'])+"))")
    if spec.get('icd9'):  p.append("(d.icd_version=9 AND ("+" OR ".join(f"d.icd_code LIKE '{c}%'" for c in spec['icd9'])+"))")
    return "("+" OR ".join(p)+")"
def group_cond(g):                       # UNION: keyword HOẶC code
    return "((" + DISEASE_TITLE[g].format(t="LOWER(dd.long_title)") + ") OR " + _code(DISEASE_CODES[g]) + ")"
_ALL = " OR ".join(group_cond(g) for g in DISEASE_TITLE)
TARGET_ADM = (f"SELECT DISTINCT d.subject_id, d.hadm_id "
              f"FROM `{H}.diagnoses_icd` d JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version) "
              f"WHERE {_ALL}")




Mounted at /content/drive


In [ ]:
# from google.colab import auth
# auth.authenticate_user()

# # cài gcsfuse
# !echo "deb https://packages.cloud.google.com/apt gcsfuse-`lsb_release -c -s` main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list >/dev/null
# !curl -s https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add - >/dev/null 2>&1
# !apt-get -qq update && apt-get -qq install -y gcsfuse >/dev/null

# # mount (requester-pays -> cần billing project của bạn)
# !mkdir -p /content/cxr_gcs
# !gcsfuse --billing-project mimic-ehr-v310 --implicit-dirs mimic-cxr-jpg-2.1.0.physionet.org /content/cxr_gcs
# !ls /content/cxr_gcs/files/p10 | head -3


W: https://packages.cloud.google.com/apt/dists/gcsfuse-jammy/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
{"timestamp":{"seconds":1783328893,"nanos":303846539},"severity":"INFO","message":"Start gcsfuse/3.10.0 (Go version go1.26.4) for app \"\" using mount point: /content/cxr_gcs\n","mount-id":"mimic-cxr-jpg-2.1.0.physionet.org-dccb1f59"}
{"timestamp":{"seconds":1783328893,"nanos":303871562},"severity":"INFO","message":"GCSFuse Config","mount-id":"mimic-cxr-jpg-2.1.0.physionet.org-dccb1f59","CLI Flags":{"billing-project":"mimic-ehr-v310","implicit-dirs":"true"}}
{"timestamp":{"seconds":1783328893,"nanos":303891071},"severity":"INFO","message":"GCSFuse Config","mount-id":"mimic-cxr-jpg-2.1.0.physionet.org-dccb1f59

In [ ]:
# from google.colab import auth
# auth.authenticate_user()
# from google.cloud import storage
# import io, time, pandas as pd
# from PIL import Image
# from concurrent.futures import ThreadPoolExecutor

# client = storage.Client(project='mimic-ehr-v310')
# bucket = client.bucket('mimic-cxr-jpg-2.1.0.physionet.org', user_project='mimic-ehr-v310')  # requester-pays

# man = pd.read_parquet('/content/drive/MyDrive/MIMIC_CXR/planA_out/manifest.parquet')
# PFX = '/content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org/'
# rels = man['jpg_path'].str.replace(PFX,'',regex=False).head(300).tolist()

# def fetch(rel):
#     b = bucket.blob(rel).download_as_bytes()
#     return Image.open(io.BytesIO(b)).convert('RGB').resize((256,256))

# for nw in [16, 48, 96]:
#     t0=time.time()
#     with ThreadPoolExecutor(max_workers=nw) as ex:
#         n=sum(1 for _ in ex.map(fetch, rels))
#     print(f"GCS client {nw} luồng: {n/(time.time()-t0):.1f} img/s")


GCS client 16 luồng: 9.6 img/s


GCS client 48 luồng: 27.1 img/s


GCS client 96 luồng: 29.0 img/s


In [ ]:
# import pandas as pd, os, re
# from google.cloud import bigquery
# client = bigquery.Client(project=BILLING_PROJECT)

# # 1) Bảng mọi mã liên quan lao + số bệnh nhân (toàn MIMIC-IV)
# tbcodes = client.query(f"""
# WITH tb AS (
#   SELECT d.subject_id, d.icd_version, d.icd_code, dd.long_title
#   FROM `{H}.diagnoses_icd` d
#   JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
#   WHERE LOWER(dd.long_title) LIKE '%tuberculosis%'
#      OR (d.icd_version=10 AND REGEXP_CONTAINS(d.icd_code, r'^A1[5-9]'))
#      OR (d.icd_version=9  AND REGEXP_CONTAINS(d.icd_code, r'^01[0-8]'))
# )
# SELECT icd_version, icd_code, ANY_VALUE(long_title) AS long_title,
#        COUNT(DISTINCT subject_id) AS n_subj
# FROM tb GROUP BY icd_version, icd_code ORDER BY n_subj DESC
# """).to_dataframe()
# pd.set_option('display.max_rows', 200, 'display.max_colwidth', 60)
# print("=== Mọi mã lao trong MIMIC-IV (theo số bệnh nhân) ===")
# print(tbcodes.to_string(index=False))

# # 2) Per-subject để giao với bệnh nhân có CXR
# tbsubj = client.query(f"""
# SELECT DISTINCT d.subject_id, d.icd_version, d.icd_code
# FROM `{H}.diagnoses_icd` d
# JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
# WHERE LOWER(dd.long_title) LIKE '%tuberculosis%'
#    OR (d.icd_version=10 AND REGEXP_CONTAINS(d.icd_code, r'^A1[5-9]'))
#    OR (d.icd_version=9  AND REGEXP_CONTAINS(d.icd_code, r'^01[0-8]'))
# """).to_dataframe()

# def is_active(v, c):
#     if v==10: return bool(re.match(r'^A1[5-9]', str(c)))
#     if v==9:  return bool(re.match(r'^01[0-8]', str(c)))
#     return False
# tbsubj['active'] = [is_active(v,c) for v,c in zip(tbsubj.icd_version, tbsubj.icd_code)]

# cxr_subj = set(pd.read_parquet(os.path.join(PLANA_DIR,'manifest.parquet')).subject_id.unique())
# tbc = tbsubj[tbsubj.subject_id.isin(cxr_subj)]

# print("\n=== LAO trong bệnh nhân CÓ CXR ===")
# print("  Lao HOẠT ĐỘNG (A15-19 / 010-018):", tbc[tbc.active].subject_id.nunique())
# print("    - qua ICD-10:", tbc[(tbc.active)&(tbc.icd_version==10)].subject_id.nunique())
# print("    - qua ICD-9 :", tbc[(tbc.active)&(tbc.icd_version==9)].subject_id.nunique())
# print("  Bất kỳ mã có 'tuberculosis' (gồm tiền sử/latent/tầm soát):", tbc.subject_id.nunique())
# tbc[['subject_id']].drop_duplicates().to_csv(os.path.join(OUT_DIR,'tb_cxr_subjects.csv'), index=False)
# print("  -> đã lưu tb_cxr_subjects.csv")


=== Mọi mã lao trong MIMIC-IV (theo số bệnh nhân) ===
 icd_version icd_code                                                                                                                                                                              long_title  n_subj
           9    V1201                                                                                                                                                        Personal history of tuberculosis     391
          10    R7611                                                                                                                Nonspecific reaction to tuberculin skin test without active tuberculosis     327
           9    79551                                                                                                                Nonspecific reaction to tuberculin skin test without active tuberculosis     315
          10    Z8611                                                                     

## 3. Cohort: CXR ∩ EHR(ICD-10)

In [ ]:
import pandas as pd, numpy as np, os
from google.cloud import bigquery

manifest = pd.read_parquet(os.path.join(PLANA_DIR, 'manifest.parquet'))
print("Manifest Plan A:", len(manifest), "ảnh |", manifest.subject_id.nunique(), "bệnh nhân")

client = bigquery.Client(project=BILLING_PROJECT)
icd10 = client.query(f"SELECT DISTINCT subject_id FROM ({TARGET_ADM})").to_dataframe()
icd10_ids = set(icd10.subject_id.tolist())
print("Bệnh nhân có ICD-10 trong EHR:", len(icd10_ids))

cohort_ids = set(manifest.subject_id.unique()) & icd10_ids
print("Giao CXR ∩ ICD-10:", len(cohort_ids), "bệnh nhân")
cohort_df = pd.DataFrame({'subject_id': sorted(cohort_ids)})
cohort_df.to_parquet(os.path.join(OUT_DIR,'cohort_subjects.parquet'), index=False)
cohort_df.to_csv(os.path.join(OUT_DIR,'cohort_subjects.csv'), index=False)
print("Đã lưu cohort_subjects.csv /.parquet:", len(cohort_df), "bệnh nhân")

Manifest Plan A: 377110 ảnh | 65379 bệnh nhân
Bệnh nhân có ICD-10 trong EHR: 80006
Giao CXR ∩ ICD-10: 26043 bệnh nhân
Đã lưu cohort_subjects.csv /.parquet: 26043 bệnh nhân


## 4. Cohort manifest + resolve nhãn — policy cho -1: **U-Ones / U-Zeros / U-Ignore** (A/B). `y_` giữ U-Ones cho fusion.

In [ ]:
# ==== Cohort manifest (cm) + Nhãn CheXpert: RAW 4 trạng thái + policy cho -1 (A/B) ====
import numpy as np, pandas as pd, os
LABEL_POLICY = 'U-Ones'   # 'U-Ones' | 'U-Zeros' | 'U-Ignore'   <- đổi để làm A/B

# --- dựng cm = manifest Plan A lọc theo cohort (tự nạp từ disk, không phụ thuộc cell trước) ---
manifest   = pd.read_parquet(os.path.join(PLANA_DIR, 'manifest.parquet'))
cohort_ids = set(pd.read_parquet(os.path.join(OUT_DIR, 'cohort_subjects.parquet')).subject_id)
cm = manifest[manifest.subject_id.isin(cohort_ids)].reset_index(drop=True)
assert all(c in cm.columns for c in CHEXPERT_LABELS), "manifest thiếu cột nhãn CheXpert thô"
print("cm (cohort manifest):", cm.shape, "|", cm.subject_id.nunique(), "bệnh nhân")

L = cm[CHEXPERT_LABELS].to_numpy('float32')   # 1 / 0 / -1(uncertain) / NaN(blank)

def make_labels(policy, L=L):
    """Trả (Y, M): Y = nhãn 0/1 để train ; M = mask (True = tính loss). blank(NaN) -> 0 luôn."""
    unc  = (L == -1.0)
    base = np.where(np.isnan(L), 0.0, L)                 # blank -> 0 (âm)
    if   policy == 'U-Ones':   Y = np.where(unc, 1.0, base); M = np.ones_like(L, bool)
    elif policy == 'U-Zeros':  Y = np.where(unc, 0.0, base); M = np.ones_like(L, bool)
    elif policy == 'U-Ignore': Y = np.where(unc, 0.0, base); M = ~unc            # -1 bị MASK
    else: raise ValueError(policy)
    return Y.astype('float32'), M

# y_ trong manifest/fusion LUÔN dùng U-Ones (feature ổn định, không NaN) — không phá downstream
Yfeat, _ = make_labels('U-Ones')
for j, c in enumerate(CHEXPERT_LABELS):
    cm['y_'+c] = Yfeat[:, j]

# KHÔNG ghi đè file gốc — chỉ ghi bản *_policy nếu chưa có
_mc = os.path.join(OUT_DIR, 'manifest_cohort_policy.parquet')
_ds = os.path.join(OUT_DIR, 'dataset_policy.csv')
if not os.path.exists(_mc): cm.to_parquet(_mc, index=False)
if not os.path.exists(_ds): cm.to_csv(_ds, index=False)
print("cm dựng xong:", len(cm), "ảnh | y_=U-Ones | ghi ra *_policy (KHÔNG đè manifest_cohort/dataset gốc)")
print(cm.split.value_counts(dropna=False))

# ==== Nhãn train theo LABEL_POLICY (dùng cho fine-tune): Y_all, M_all ====
Y_all, M_all = make_labels(LABEL_POLICY)

# ==== Thống kê 1/0/-1/blank mỗi nhãn (TRAIN) + số thực dùng theo policy ====
tr = (cm.split == 'train').to_numpy()
rows = []
for j, c in enumerate(CHEXPERT_LABELS):
    col = L[tr, j]
    pos = int((col == 1).sum()); neg = int((col == 0).sum())
    unc = int((col == -1).sum()); blk = int(np.isnan(col).sum())
    ytr, mtr = Y_all[tr, j], M_all[tr, j]
    used_pos = int((ytr[mtr] == 1).sum()); used_neg = int((ytr[mtr] == 0).sum())
    rows.append({'label': c, 'pos_1': pos, 'neg_0': neg, 'uncertain_-1': unc, 'blank': blk,
                 f'{LABEL_POLICY}_1': used_pos, f'{LABEL_POLICY}_0': used_neg,
                 f'{LABEL_POLICY}_masked': int((~mtr).sum())})
stat = pd.DataFrame(rows).set_index('label')
print(f"\n[LABEL_POLICY = {LABEL_POLICY}] thống kê nhãn trên TRAIN:")
print(stat.to_string())

In [ ]:
from google.cloud import bigquery
import pandas as pd, os
client = bigquery.Client(project=BILLING_PROJECT)

GROUPS = list(DISEASE_TITLE.keys())
dx_sel = ",\n  ".join(f"MAX(CASE WHEN {group_cond(g)} THEN 1 ELSE 0 END) AS dx_{g}" for g in GROUPS)
dxg = client.query(f"""
WITH cohort AS (SELECT DISTINCT subject_id FROM ({TARGET_ADM}))
SELECT d.subject_id,
  {dx_sel}
FROM `{H}.diagnoses_icd` d
JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
JOIN cohort c USING (subject_id)
GROUP BY d.subject_id
""").to_dataframe()

dxcols = ['dx_'+g for g in GROUPS]
dxg['disease_group'] = dxg[dxcols].apply(lambda r: '|'.join(g for g,v in zip(GROUPS, r.values) if v==1), axis=1)
print("Số bệnh nhân / nhóm bệnh (long_title, ICD-9+10, mọi lần nằm viện):")
for g in GROUPS: print(f"  {g:<16}: {int(dxg['dx_'+g].sum())}")
print(dxg['disease_group'].value_counts().head(15))

def add_group(path, kind):
    if not os.path.exists(path): print("bỏ qua (chưa có):", os.path.basename(path)); return
    df = pd.read_csv(path)
    df = df.drop(columns=[c for c in dxcols+['disease_group'] if c in df.columns]).merge(dxg, on='subject_id', how='left')
    df.to_csv(path, index=False); print(f"Đã cập nhật {os.path.basename(path)} ({kind}):", df.shape)

add_group(os.path.join(OUT_DIR,'dataset.csv'),        'ảnh-level')
add_group(os.path.join(OUT_DIR,'fusion_dataset.csv'), 'bệnh nhân-level')
mc = os.path.join(OUT_DIR,'manifest_cohort.parquet')
if os.path.exists(mc):
    m = pd.read_parquet(mc)
    m = m.drop(columns=[c for c in dxcols+['disease_group'] if c in m.columns]).merge(dxg, on='subject_id', how='left')
    m.to_parquet(mc, index=False); print("Đã cập nhật manifest_cohort.parquet:", m.shape)


Số bệnh nhân / nhóm bệnh (long_title, ICD-9+10, mọi lần nằm viện):
  pneumonia       : 31114
  bronchitis      : 3792
  tuberculosis    : 2034
  arthritis       : 24251
  fracture        : 32394
  scoliosis       : 1607
  lung_cancer     : 9320
  lung_infection  : 5437
disease_group
fracture                        20627
pneumonia                       15568
arthritis                       13412
lung_cancer                      3997
pneumonia|fracture               3531
arthritis|fracture               3294
pneumonia|arthritis              2725
pneumonia|lung_cancer            1719
lung_infection                   1353
pneumonia|lung_infection         1348
pneumonia|arthritis|fracture     1122
bronchitis                        935
tuberculosis                      791
pneumonia|bronchitis              782
fracture|lung_cancer              756
Name: count, dtype: int64
Đã cập nhật dataset.csv (ảnh-level): (231684, 44)
bỏ qua (chưa có): fusion_dataset.csv
Đã cập nhật manifest_cohort.parqu

In [ ]:
import pandas as pd, os
GROUPS = list(DISEASE_TITLE.keys())
df = pd.read_csv(os.path.join(OUT_DIR,'dataset.csv')).drop_duplicates('subject_id')
print("Cohort CXR∩EHR:", len(df), "bệnh nhân")
for g in GROUPS: print(f"  {g:<16}: {int(df['dx_'+g].sum())}")
print(df['disease_group'].value_counts().head(15))


Cohort CXR∩EHR: 26043 bệnh nhân
  pneumonia       : 12788
  bronchitis      : 2088
  tuberculosis    : 955
  arthritis       : 8936
  fracture        : 9640
  scoliosis       : 526
  lung_cancer     : 3328
  lung_infection  : 2535
disease_group
pneumonia                       5281
fracture                        4234
arthritis                       3754
pneumonia|fracture              1574
pneumonia|arthritis             1321
arthritis|fracture              1160
lung_cancer                     1151
pneumonia|arthritis|fracture     691
pneumonia|lung_cancer            622
lung_infection                   531
pneumonia|lung_infection         474
pneumonia|bronchitis             392
bronchitis                       364
tuberculosis                     288
fracture|lung_cancer             257
Name: count, dtype: int64


## 5. Cache ảnh 256px một lần (resumable)\n\nĐọc mỗi ảnh cohort từ Drive **một lần**, resize 256 RGB uint8, ghi shard trên Drive. Chạy lại thì skip ảnh đã cache. Đây là bước tốn I/O nhất — chạy nhiều phiên nếu cần.

## 6. Copy cache về local + Dataset + normalize\n\nCopy file .npy lớn từ Drive về `/content` (nhanh vì file lớn), memmap để train đọc ngẫu nhiên tốc độ cao.

In [ ]:
from google.colab import auth; auth.authenticate_user()
from google.cloud import storage
import google.auth
from google.auth.transport.requests import AuthorizedSession
from requests.adapters import HTTPAdapter
from concurrent.futures import ThreadPoolExecutor
import io, os, glob, time, numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import AutoModel, AutoImageProcessor

REPO='microsoft/rad-dino'; BILLING_PROJECT='mimic-ehr-v310'
OUT_DIR='/content/drive/MyDrive/MIMIC_CXR/planB_out'; TRAIN_RES=224; BATCH=64
BUCKET='mimic-cxr-jpg-2.1.0.physionet.org'; THREADS=32

device=torch.device('cuda')
model=AutoModel.from_pretrained(REPO).eval().to(device)
for p in model.parameters(): p.requires_grad=False
proc=AutoImageProcessor.from_pretrained(REPO)
tf=T.Compose([T.Resize(TRAIN_RES),T.CenterCrop(TRAIN_RES),T.ToTensor(),
              T.Normalize(proc.image_mean,proc.image_std)])

cm=pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))
cm['rel']=cm['jpg_path'].str.split('mimic-cxr-jpg-2.1.0.physionet.org/').str[-1]
EMB_C=[f'cls_{i}' for i in range(768)]; EMB_M=[f'mean_{i}' for i in range(768)]
SHARD_DIR=os.path.join(OUT_DIR,'emb_shards'); os.makedirs(SHARD_DIR, exist_ok=True)

_bucket=None
def _get_bucket():
    global _bucket
    if _bucket is None:
        creds,_ = google.auth.default()
        sess = AuthorizedSession(creds)
        sess.mount('https://', HTTPAdapter(pool_connections=THREADS, pool_maxsize=THREADS))
        _bucket = storage.Client(project=BILLING_PROJECT, credentials=creds, _http=sess
                                 ).bucket(BUCKET, user_project=BILLING_PROJECT)   # <-- credentials=creds
    return _bucket

def _one(rel):
    try: return tf(Image.open(io.BytesIO(_get_bucket().blob(rel).download_as_bytes())).convert('RGB'))
    except Exception: return None

class BatchGcsDS(Dataset):
    def __init__(self, df, bs): self.rel=df['rel'].tolist(); self.id=df['dicom_id'].tolist(); self.bs=bs
    def __len__(self): return (len(self.id)+self.bs-1)//self.bs
    def __getitem__(self, b):
        lo=b*self.bs; hi=min(lo+self.bs,len(self.id)); ids=self.id[lo:hi]; rels=self.rel[lo:hi]
        with ThreadPoolExecutor(THREADS) as ex: ims=list(ex.map(_one, rels))
        keep=[(i,im) for i,im in zip(ids,ims) if im is not None]
        if not keep: return [], None
        return [k[0] for k in keep], torch.stack([k[1] for k in keep])
print("OK. cohort:", len(cm))


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

OK. cohort: 231684


In [ ]:
print(len(_get_bucket().blob(cm['rel'].iloc[0]).download_as_bytes()), "bytes")


1707658 bytes


## 7. Model RAD-DINO + fine-tune last-N blocks

In [ ]:
from tqdm.auto import tqdm
done=set()
for f in glob.glob(os.path.join(SHARD_DIR,'*.parquet')):
    done |= set(pd.read_parquet(f, columns=['dicom_id']).dicom_id.tolist())
todo = cm[~cm.dicom_id.isin(done)].reset_index(drop=True)
print("Đã trích:", len(done), "| còn:", len(todo))

loader = DataLoader(BatchGcsDS(todo, BATCH), batch_size=None, num_workers=4, prefetch_factor=2)
sh = len(glob.glob(os.path.join(SHARD_DIR,'*.parquet')))
buf_id, buf_c, buf_m, seen, t0 = [],[],[],0,time.time()
SAVE_EVERY = 4000
def flush():
    global sh, buf_id, buf_c, buf_m
    df = pd.concat([pd.DataFrame(np.vstack(buf_c),columns=EMB_C),
                    pd.DataFrame(np.vstack(buf_m),columns=EMB_M)], axis=1)
    df.insert(0,'dicom_id',buf_id)
    df.to_parquet(os.path.join(SHARD_DIR,f'emb_{sh:05d}.parquet'), index=False); sh+=1
    buf_id, buf_c, buf_m = [],[],[]
pbar=tqdm(loader, total=len(loader))
for ids, x in pbar:
    if x is None: continue
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        h=model(pixel_values=x.to(device), interpolate_pos_encoding=True).last_hidden_state
        c=h[:,0].float().cpu().numpy(); m=h[:,1:].mean(1).float().cpu().numpy()
    buf_id+=ids; buf_c.append(c); buf_m.append(m); seen+=len(ids)
    pbar.set_postfix(img_s=round(seen/(time.time()-t0),1), done=seen)
    if sum(a.shape[0] for a in buf_c) >= SAVE_EVERY: flush()
if buf_id: flush()
print("XONG:", seen, "| shard:", sh)


Đã trích: 210989 | còn: 20695


  0%|          | 0/324 [00:00<?, ?it/s]

XONG: 20695 | shard: 59


In [ ]:
files = sorted(glob.glob(os.path.join(SHARD_DIR,'*.parquet')))
alldf = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True).drop_duplicates('dicom_id')
idx = cm[cm.dicom_id.isin(set(alldf.dicom_id))].reset_index(drop=True)
alldf = alldf.set_index('dicom_id').loc[idx.dicom_id].reset_index()
np.save(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'),      alldf[EMB_C].to_numpy('float32'))
np.save(os.path.join(OUT_DIR,'embeddings_raddino_ft_meanpool.npy'), alldf[EMB_M].to_numpy('float32'))
out = idx[['dicom_id','subject_id','study_id','split','view']].copy(); out['row_index']=np.arange(len(out))
out.to_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'), index=False)
print("Frozen embeddings:", len(alldf), "ảnh × 768 (CLS + mean-pool)")


Frozen embeddings: 231684 ảnh × 768 (CLS + mean-pool)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

emb = np.load(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'))
idx = pd.read_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'))
lab = cm[['dicom_id']+['y_'+c for c in CHEXPERT_LABELS]]
d = idx.merge(lab, on='dicom_id', how='left')
tr, te = (d.split=='train').values, (d.split=='test').values
sc = StandardScaler().fit(emb[tr]); Xtr, Xte = sc.transform(emb[tr]), sc.transform(emb[te])
rows=[]
for c in CHEXPERT_LABELS:
    ytr=d.loc[tr,'y_'+c].values; yte=d.loc[te,'y_'+c].values
    if len(np.unique(ytr))<2 or len(np.unique(yte))<2: rows.append((c,np.nan,np.nan,np.nan)); continue
    clf=LogisticRegression(max_iter=1000).fit(Xtr, ytr); p=clf.predict_proba(Xte)[:,1]
    rows.append((c, roc_auc_score(yte,p), precision_score(yte,p>=.5,zero_division=0), recall_score(yte,p>=.5,zero_division=0)))
met=pd.DataFrame(rows, columns=['label','AUROC','precision','recall'])
met.to_csv(os.path.join(OUT_DIR,'raddino_frozen_linearprobe.csv'), index=False)
print(met.to_string(index=False)); print("Macro AUROC:", round(met.AUROC.mean(),4))


                     label    AUROC  precision   recall
               Atelectasis 0.746532   0.534247 0.240317
              Cardiomegaly 0.760954   0.564103 0.290669
             Consolidation 0.684992   1.000000 0.004640
                     Edema 0.818109   0.630229 0.386589
Enlarged Cardiomediastinum 0.666731   0.000000 0.000000
                  Fracture 0.690377   0.000000 0.000000
               Lung Lesion 0.718607   0.125000 0.005000
              Lung Opacity 0.657061   0.575980 0.155939
                No Finding 0.762026   0.553753 0.322314
          Pleural Effusion 0.846014   0.716381 0.596741
             Pleural Other 0.776181   0.000000 0.000000
                 Pneumonia 0.675176   0.585366 0.071429
              Pneumothorax 0.812057   0.358025 0.190789
           Support Devices 0.866856   0.718090 0.608560
Macro AUROC: 0.7487


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import numpy as np, pandas as pd, os

emb = np.load(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'))
idx = pd.read_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'))
cm  = pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))
d = idx.merge(cm[['dicom_id']+['y_'+c for c in CHEXPERT_LABELS]], on='dicom_id', how='left')
tr,va,te = (d.split=='train').values,(d.split=='validate').values,(d.split=='test').values
sc = StandardScaler().fit(emb[tr]); Xtr,Xva,Xte = sc.transform(emb[tr]),sc.transform(emb[va]),sc.transform(emb[te])
rows=[]
for c in CHEXPERT_LABELS:
    ytr,yva,yte = d.loc[tr,'y_'+c].values, d.loc[va,'y_'+c].values, d.loc[te,'y_'+c].values
    if len(np.unique(ytr))<2: rows.append((c,np.nan,np.nan,np.nan,np.nan)); continue
    clf=LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr,ytr)
    pva,pte = clf.predict_proba(Xva)[:,1], clf.predict_proba(Xte)[:,1]
    ths=np.linspace(0.05,0.95,19)
    bt = max(ths, key=lambda t: f1_score(yva, pva>=t, zero_division=0)) if len(np.unique(yva))==2 else 0.5
    pred = pte>=bt
    au = roc_auc_score(yte,pte) if len(np.unique(yte))==2 else np.nan
    rows.append((c, au, precision_score(yte,pred,zero_division=0),
                 recall_score(yte,pred,zero_division=0), f1_score(yte,pred,zero_division=0)))
met=pd.DataFrame(rows, columns=['label','AUROC','precision','recall','F1'])
print(met.to_string(index=False)); print("Macro AUROC:",round(met.AUROC.mean(),4),"| Macro F1:",round(met.F1.mean(),4))


                     label    AUROC  precision   recall       F1
               Atelectasis 0.746433   0.413408 0.716549 0.524316
              Cardiomegaly 0.759711   0.398924 0.857143 0.544453
             Consolidation 0.688721   0.200675 0.276102 0.232422
                     Edema 0.817532   0.514252 0.731788 0.604031
Enlarged Cardiomediastinum 0.666772   0.138794 0.680288 0.230550
                  Fracture 0.693412   0.122807 0.239726 0.162413
               Lung Lesion 0.718182   0.128378 0.285000 0.177019
              Lung Opacity 0.657714   0.408056 0.773059 0.534159
                No Finding 0.762335   0.428841 0.537190 0.476939
          Pleural Effusion 0.846332   0.629547 0.763747 0.690184
             Pleural Other 0.772181   0.079710 0.592308 0.140511
                 Pneumonia 0.674780   0.319708 0.651786 0.428991
              Pneumothorax 0.807209   0.176471 0.434211 0.250951
           Support Devices 0.867431   0.651964 0.736187 0.691520
Macro AUROC: 0.7485 | Mac

## 8. Đánh giá 14-label (AUROC + precision/recall/F1) + heatmap

In [ ]:
# import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
# from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

# ck = torch.load(os.path.join(OUT_DIR,'raddino_ft_best.pt'), map_location=device)
# model.load_state_dict(ck['model']); model.eval()

# _, Pv, Yv = evaluate(va_loader)
# te_loader = DataLoader(CacheDS(te_idx, False), batch_size=BATCH, shuffle=False, num_workers=8, pin_memory=True)
# _, Pt, Yt = evaluate(te_loader)

# rows=[]
# for j,c in enumerate(CHEXPERT_LABELS):
#     if len(np.unique(Yv[:,j]))<2 or len(np.unique(Yt[:,j]))<2:
#         rows.append((c,np.nan,np.nan,np.nan,np.nan)); continue
#     ths=np.linspace(0.05,0.95,19)
#     bt=max(ths, key=lambda t: f1_score(Yv[:,j], Pv[:,j]>=t, zero_division=0))   # threshold theo F1 trên val
#     pr=Pt[:,j]>=bt
#     rows.append((c, roc_auc_score(Yt[:,j],Pt[:,j]),
#                  precision_score(Yt[:,j],pr,zero_division=0),
#                  recall_score(Yt[:,j],pr,zero_division=0),
#                  f1_score(Yt[:,j],pr,zero_division=0)))
# met=pd.DataFrame(rows, columns=['label','AUROC','precision','recall','F1'])
# met.to_csv(os.path.join(OUT_DIR,'raddino_metrics.csv'), index=False)
# print(met.to_string(index=False))
# print("Macro AUROC:", round(met.AUROC.mean(),4), "| Macro F1:", round(met.F1.mean(),4))

# fig,ax=plt.subplots(figsize=(6,7))
# sns.heatmap(met.set_index('label')[['AUROC','precision','recall','F1']], annot=True, fmt='.2f',
#             cmap='viridis', vmin=0, vmax=1, ax=ax)
# ax.set_title('RAD-DINO fine-tuned — 14-label (test)')
# plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,'raddino_metrics_heatmap.png'), dpi=120); plt.show()

## 8-bis. A/B policy nhãn -1 (FROZEN) — U-Ones / U-Zeros / U-Ignore trên test 0/1 cố định

Dùng embedding frozen (`embeddings_raddino_ft_cls.npy`) + nhãn RAW 4 trạng thái. Train 1 LogReg/nhãn/policy; test & val CHỈ nhãn chắc chắn (0/1) nên 3 policy chấm trên cùng ground truth. Chọn policy thắng theo macro-AUROC / macro-F2 rồi mới fine-tune.

In [ ]:
# ===== A/B policy trên FROZEN feature (test 0/1 cố định) =====
import numpy as np, pandas as pd, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, fbeta_score, precision_score

emb = np.load(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'))
idx = pd.read_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'))
cm  = pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))

FRONTAL_ONLY = True
m = idx[['dicom_id']].merge(cm[['dicom_id','split','view']+CHEXPERT_LABELS], on='dicom_id', how='left')
keep_row = (m.view=='frontal').values if FRONTAL_ONLY else np.ones(len(m), bool)
L = m[CHEXPERT_LABELS].to_numpy('float32')          # 1/0/-1/NaN
split = m['split'].values
tr = (split=='train')&keep_row; va = (split=='validate')&keep_row; te = (split=='test')&keep_row
print("train/val/test (frontal=%s):"%FRONTAL_ONLY, int(tr.sum()), int(va.sum()), int(te.sum()))

X = StandardScaler().fit(emb[tr]).transform(emb).astype('float32')

def resolve(policy, col):
    unc=col==-1; base=np.where(np.isnan(col),0.0,col)
    if policy=='U-Ones':   return np.where(unc,1.0,base).astype(int), np.ones(len(col),bool)
    if policy=='U-Zeros':  return np.where(unc,0.0,base).astype(int), np.ones(len(col),bool)
    if policy=='U-Ignore': return np.where(unc,0.0,base).astype(int), ~unc

GRID=np.arange(0.05,0.96,0.01)
def best_thr(y,p,beta=2):
    s=[fbeta_score(y,(p>=t).astype(int),beta=beta,zero_division=0) for t in GRID]
    return float(GRID[int(np.argmax(s))])

rows=[]
for pol in ['U-Ones','U-Zeros','U-Ignore']:
    for j,c in enumerate(CHEXPERT_LABELS):
        col=L[:,j]; y,keep=resolve(pol,col)
        trm=tr&keep                                 # U-Ignore: bỏ -1 khỏi train
        certain=(col==0)|(col==1)                   # test/val CỐ ĐỊNH: chỉ nhãn chắc chắn
        vam, tem = va&certain, te&certain
        clf=LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1).fit(X[trm], y[trm])
        pv=clf.predict_proba(X[vam])[:,1]; pt=clf.predict_proba(X[tem])[:,1]
        yv=col[vam].astype(int); yt=col[tem].astype(int)
        thr=best_thr(yv,pv,beta=2); pred=(pt>=thr).astype(int)
        au=roc_auc_score(yt,pt) if 0<yt.sum()<len(yt) else np.nan
        rows.append(dict(policy=pol, label=c, auroc=au,
            recall=recall_score(yt,pred,zero_division=0),
            f2=fbeta_score(yt,pred,beta=2,zero_division=0),
            precision=precision_score(yt,pred,zero_division=0),
            thr=round(thr,3), pos_train=int(y[trm].sum()), n_train=int(trm.sum()),
            pos_test=int(yt.sum()), n_test=int(tem.sum())))
        print(f"[{pol:9}] {c:28} AUROC={au:.3f} recall={rows[-1]['recall']:.3f} F2={rows[-1]['f2']:.3f}")

res=pd.DataFrame(rows); res.to_csv(os.path.join(OUT_DIR,'ab_policy_frozen.csv'), index=False)
print("\nlưu ab_policy_frozen.csv | dòng:", len(res), "(3x14=42)")

In [ ]:
# ===== Chọn policy thắng + heatmap =====
import pandas as pd, numpy as np, os, matplotlib.pyplot as plt, seaborn as sns
res=pd.read_csv(os.path.join(OUT_DIR,'ab_policy_frozen.csv'))
POLS=['U-Ones','U-Zeros','U-Ignore']

macro = res.groupby('policy')[['auroc','recall','f2','precision']].mean().reindex(POLS).round(4)
print("== MACRO (mean 14 nhãn) =="); print(macro)
print("\nPolicy tốt nhất — macro-AUROC:", macro['auroc'].idxmax(), "| macro-F2:", macro['f2'].idxmax())

for metric in ['auroc','recall','f2']:
    piv=res.pivot(index='label', columns='policy', values=metric).reindex(columns=POLS)
    plt.figure(figsize=(6,6))
    sns.heatmap(piv, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
                linewidths=.5, linecolor='black', annot_kws={'size':8}, cbar_kws={'label':metric})
    plt.title(f'A/B policy - {metric} (test 0/1 co dinh)'); plt.ylabel('nhan'); plt.xlabel('')
    plt.yticks(rotation=0); plt.tight_layout(); plt.show()

## 9. Trích embedding 768-dim (CLS + mean-pool) từ backbone fine-tuned

In [ ]:
# import numpy as np, pandas as pd, torch

# all_idx = gmeta.index.to_numpy()
# emb_loader = DataLoader(CacheDS(all_idx, False), batch_size=BATCH, shuffle=False, num_workers=8, pin_memory=True)
# model.eval()
# CLS, MEANP = [], []
# with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
#     for x,_ in emb_loader:
#         cls, mp = model(x.to(device), return_emb=True)
#         CLS.append(cls.float().cpu().numpy()); MEANP.append(mp.float().cpu().numpy())
# CLS=np.concatenate(CLS); MEANP=np.concatenate(MEANP)
# np.save(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'), CLS)
# np.save(os.path.join(OUT_DIR,'embeddings_raddino_ft_meanpool.npy'), MEANP)

# idx = gmeta[['dicom_id','split']].copy(); idx['row_index']=np.arange(len(idx))
# cm2 = pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))[['dicom_id','subject_id','study_id','view']]
# idx = idx.merge(cm2, on='dicom_id', how='left')
# idx.to_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'), index=False)
# print("Embeddings CLS:", CLS.shape, "| mean-pool:", MEANP.shape)
# print("NaN CLS:", bool(np.isnan(CLS).any()), "| NaN mean:", bool(np.isnan(MEANP).any()))
# print("Đã lưu vào", OUT_DIR)

# MIMIC-ED Query

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project='mimic-ehr-v310')
# tên dataset ED (không có _3_1); nếu lỗi thử liệt kê
df = client.query("""
  SELECT * FROM `physionet-data.mimiciv_ed.edstays` LIMIT 5
""").to_dataframe()
print(df.columns.tolist()); df.head()


['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'gender', 'race', 'arrival_transport', 'disposition']


,subject_id,hadm_id,stay_id,intime,outtime,gender,race,arrival_transport,disposition
0,10000178,<NA>,31721172,2157-04-08 09:58:00,2157-04-08 15:30:00,F,ASIAN,WALK IN,HOME
1,10004371,<NA>,38482952,2182-05-14 14:08:00,2182-05-14 16:14:00,F,ASIAN,WALK IN,HOME
2,10005303,<NA>,32459569,2135-07-05 12:10:00,2135-07-05 16:07:00,M,ASIAN,WALK IN,HOME
3,10005329,25902419,32044805,2138-08-11 19:09:00,2138-08-12 08:28:00,F,ASIAN,WALK IN,HOME
4,10009866,<NA>,37316180,2123-01-28 16:36:00,2123-01-28 19:03:00,M,ASIAN,WALK IN,HOME


## 10. Feature EHR tại lần nhập viện ICD-10 đầu tiên (demographics + vitals + labs + microbiology + anchor)

In [ ]:
# from google.cloud import bigquery
# import pandas as pd, numpy as np, os
# client = bigquery.Client(project=BILLING_PROJECT)
# H = "physionet-data.mimiciv_3_1_hosp"
# I = "physionet-data.mimiciv_3_1_icu"

# FA = f"""
#   WITH icd10_adm AS ({TARGET_ADM}),
#   first_adm AS (
#     SELECT a.subject_id, a.hadm_id,
#            ROW_NUMBER() OVER (PARTITION BY a.subject_id ORDER BY a.admittime, a.hadm_id) rn
#     FROM `{H}.admissions` a JOIN icd10_adm i USING (subject_id, hadm_id))
#   SELECT subject_id, hadm_id FROM first_adm WHERE rn=1
# """

# # 0) anchor EHR (bản gốc chưa có sẵn trong manifest)
# anchor = client.query(f"""
# WITH fa AS ({FA})
# SELECT fa.subject_id, fa.hadm_id AS first_hadm_id, a.admittime AS first_admittime,
#        STRING_AGG(d.icd_code, '|' ORDER BY d.seq_num) AS first_icd10_codes
# FROM fa
# JOIN `{H}.admissions` a ON a.hadm_id=fa.hadm_id
# LEFT JOIN `{H}.diagnoses_icd` d ON d.subject_id=fa.subject_id AND d.hadm_id=fa.hadm_id AND d.icd_version=10
# GROUP BY fa.subject_id, fa.hadm_id, a.admittime
# """).to_dataframe()
# print("anchor:", anchor.shape)

# # 1) demographics + admission
# demo = client.query(f"""
# WITH fa AS ({FA})
# SELECT fa.subject_id, p.anchor_age AS age, p.gender,
#        a.admission_type, a.insurance, a.race, a.marital_status
# FROM fa
# JOIN `{H}.patients`   p ON p.subject_id=fa.subject_id
# JOIN `{H}.admissions` a ON a.hadm_id=fa.hadm_id
# """).to_dataframe()
# print("demo:", demo.shape)

# # 2) labs (mean)
# labs = client.query(f"""
# WITH fa AS ({FA}),
# lab AS (
#   SELECT fa.subject_id,
#     CASE
#       WHEN dl.label='Hemoglobin'        THEN 'hemoglobin'
#       WHEN dl.label='White Blood Cells' THEN 'wbc'
#       WHEN dl.label='Red Blood Cells'   THEN 'rbc'
#       WHEN dl.label='Platelet Count'    THEN 'platelet'
#       WHEN dl.label='Creatinine'        THEN 'creatinine'
#       WHEN dl.label='Sodium'            THEN 'sodium'
#       WHEN dl.label='Potassium'         THEN 'potassium'
#       WHEN dl.label='Glucose'           THEN 'glucose'
#       WHEN dl.label='Urea Nitrogen'     THEN 'bun'
#       WHEN dl.label='Chloride'          THEN 'chloride'
#     END AS feat, le.valuenum
#   FROM fa
#   JOIN `{H}.labevents`  le ON le.hadm_id=fa.hadm_id
#   JOIN `{H}.d_labitems` dl ON dl.itemid=le.itemid
#   WHERE le.valuenum IS NOT NULL)
# SELECT subject_id, feat, AVG(valuenum) v FROM lab WHERE feat IS NOT NULL GROUP BY subject_id, feat
# """).to_dataframe()
# labs = labs.pivot(index='subject_id', columns='feat', values='v').add_prefix('lab_').reset_index()
# print("labs:", labs.shape)

# # 3) vitals ICU (mean); temperature -> Celsius
# vit = client.query(f"""
# WITH fa AS ({FA}),
# v AS (
#   SELECT fa.subject_id,
#     CASE WHEN di.label='Heart Rate' THEN 'heart_rate'
#          WHEN di.label='O2 saturation pulseoxymetry' THEN 'spo2' END AS feat, ce.valuenum AS val
#   FROM fa JOIN `{I}.chartevents` ce ON ce.hadm_id=fa.hadm_id
#           JOIN `{I}.d_items` di ON di.itemid=ce.itemid
#   WHERE ce.valuenum IS NOT NULL AND di.label IN ('Heart Rate','O2 saturation pulseoxymetry')
#   UNION ALL
#   SELECT fa.subject_id, 'temperature' AS feat,
#     CASE WHEN di.label='Temperature Celsius'    THEN ce.valuenum
#          WHEN di.label='Temperature Fahrenheit' THEN (ce.valuenum-32)*5/9 END AS val
#   FROM fa JOIN `{I}.chartevents` ce ON ce.hadm_id=fa.hadm_id
#           JOIN `{I}.d_items` di ON di.itemid=ce.itemid
#   WHERE ce.valuenum IS NOT NULL AND di.label IN ('Temperature Celsius','Temperature Fahrenheit'))
# SELECT subject_id, feat, AVG(val) v FROM v WHERE feat IS NOT NULL AND val IS NOT NULL GROUP BY subject_id, feat
# """).to_dataframe()
# vit = vit.pivot(index='subject_id', columns='feat', values='v').add_prefix('vital_').reset_index()
# print("vitals:", vit.shape)

# # 4) microbiology (long)
# micro = client.query(f"""
# WITH fa AS ({FA})
# SELECT fa.subject_id, m.spec_type_desc AS specimen, m.org_name
# FROM fa JOIN `{H}.microbiologyevents` m ON m.hadm_id=fa.hadm_id
# """).to_dataframe()
# print("micro rows:", micro.shape, "| org_name khác NULL:", int(micro.org_name.notna().sum()))

# # 5) cờ smoke / cough suy từ ICD (mọi lần khám của bệnh nhân)
# flags = client.query(f"""
# WITH fa AS ({FA})
# SELECT d.subject_id,
#   MAX(CASE WHEN (d.icd_version=10 AND (d.icd_code LIKE 'F17%' OR d.icd_code IN ('Z720','Z87891')))
#              OR (d.icd_version=9  AND d.icd_code IN ('3051','V1582')) THEN 1 ELSE 0 END) AS smoke,
#   MAX(CASE WHEN (d.icd_version=10 AND d.icd_code LIKE 'R05%')
#              OR (d.icd_version=9  AND d.icd_code='7862') THEN 1 ELSE 0 END) AS cough
# FROM `{H}.diagnoses_icd` d
# JOIN (SELECT DISTINCT subject_id FROM fa) s ON s.subject_id=d.subject_id
# GROUP BY d.subject_id
# """).to_dataframe()
# print("flags smoke/cough:", flags.shape, "| smoke:", int(flags.smoke.sum()), "| cough:", int(flags.cough.sum()))

In [ ]:
import pandas as pd, numpy as np, os
from google.cloud import bigquery
client = bigquery.Client(project=BILLING_PROJECT)
H="physionet-data.mimiciv_3_1_hosp"; I="physionet-data.mimiciv_3_1_icu"; ED="physionet-data.mimiciv_ed"

MODE = 'encounter'   # 'encounter' | 'firstadm'

cm  = pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))
COH = set(cm.subject_id.unique())

if MODE == 'encounter':
    meta = pd.read_csv(os.path.join(BASE,'mimic-cxr-2.0.0-metadata.csv.gz'),
                       usecols=['dicom_id','subject_id','study_id','StudyDate','StudyTime'])
    meta = meta[meta.dicom_id.isin(set(cm.dicom_id))].copy()
    _dd=meta.StudyDate.astype('int64').astype(str).str.zfill(8)
    _tt=meta.StudyTime.fillna(0).astype(float).astype('int64').astype(str).str.zfill(6).str.slice(0,6)
    meta['study_dt']=pd.to_datetime(_dd+_tt, format='%Y%m%d%H%M%S', errors='coerce')
    stt = meta[['dicom_id','subject_id','study_id','study_dt']].dropna(subset=['study_dt']).reset_index(drop=True)
    adm = client.query(f"SELECT subject_id,hadm_id,admittime,dischtime FROM `{H}.admissions`").to_dataframe()
    adm = adm[adm.subject_id.isin(COH)]; adm['admittime']=pd.to_datetime(adm.admittime); adm['dischtime']=pd.to_datetime(adm.dischtime)
    eds = client.query(f"SELECT subject_id,stay_id,hadm_id,intime,outtime FROM `{ED}.edstays`").to_dataframe()
    eds = eds[eds.subject_id.isin(COH)]; eds['intime']=pd.to_datetime(eds.intime); eds['outtime']=pd.to_datetime(eds.outtime)
    ji = stt.merge(adm,on='subject_id'); ji = ji[(ji.study_dt>=ji.admittime)&(ji.study_dt<=ji.dischtime)]
    ji = ji.sort_values('admittime').drop_duplicates('dicom_id')[['dicom_id','hadm_id']]
    je = stt.merge(eds,on='subject_id'); je = je[(je.study_dt>=je.intime)&(je.study_dt<=je.outtime)]
    je = je.sort_values('intime').drop_duplicates('dicom_id')[['dicom_id','stay_id','hadm_id']].rename(columns={'hadm_id':'ed_hadm'})
    link = stt[['dicom_id','subject_id','study_id']].merge(ji,on='dicom_id',how='left').merge(je,on='dicom_id',how='left')
    link['hadm_id'] = link['hadm_id'].fillna(link['ed_hadm'])
else:
    fa = client.query(f"""WITH ta AS ({TARGET_ADM}),
      fadm AS (SELECT a.subject_id,a.hadm_id, ROW_NUMBER() OVER (PARTITION BY a.subject_id ORDER BY a.admittime,a.hadm_id) rn
               FROM `{H}.admissions` a JOIN ta USING(subject_id,hadm_id))
      SELECT subject_id,hadm_id FROM fadm WHERE rn=1""").to_dataframe()
    link = cm[['dicom_id','subject_id','study_id']].drop_duplicates('dicom_id').merge(fa,on='subject_id',how='left'); link['stay_id']=np.nan
print(f"[{MODE}] CXR {len(link)} | hadm {link.hadm_id.notna().sum()} | ED {int(link.stay_id.notna().sum())} | none {(link.hadm_id.isna()&link.stay_id.isna()).sum()}")

CO = f"WITH cohort AS (SELECT DISTINCT subject_id FROM ({TARGET_ADM}))"
demo = client.query(f"{CO} SELECT p.subject_id,p.anchor_age age,p.gender FROM `{H}.patients` p JOIN cohort USING(subject_id)").to_dataframe()
labs = client.query(f"""{CO}
SELECT le.hadm_id,
 AVG(IF(dl.label='Hemoglobin',le.valuenum,NULL)) lab_hemoglobin, AVG(IF(dl.label='White Blood Cells',le.valuenum,NULL)) lab_wbc,
 AVG(IF(dl.label='Red Blood Cells',le.valuenum,NULL)) lab_rbc, AVG(IF(dl.label='Platelet Count',le.valuenum,NULL)) lab_platelet,
 AVG(IF(dl.label='Creatinine',le.valuenum,NULL)) lab_creatinine, AVG(IF(dl.label='Sodium',le.valuenum,NULL)) lab_sodium,
 AVG(IF(dl.label='Potassium',le.valuenum,NULL)) lab_potassium, AVG(IF(dl.label='Glucose',le.valuenum,NULL)) lab_glucose,
 AVG(IF(dl.label='Urea Nitrogen',le.valuenum,NULL)) lab_bun, AVG(IF(dl.label='Chloride',le.valuenum,NULL)) lab_chloride
FROM `{H}.labevents` le JOIN cohort c ON c.subject_id=le.subject_id JOIN `{H}.d_labitems` dl ON dl.itemid=le.itemid
WHERE le.valuenum IS NOT NULL AND le.hadm_id IS NOT NULL GROUP BY le.hadm_id""").to_dataframe()
vicu = client.query(f"""{CO}
SELECT ce.hadm_id, AVG(IF(di.label='Heart Rate',ce.valuenum,NULL)) hr_icu, AVG(IF(di.label='O2 saturation pulseoxymetry',ce.valuenum,NULL)) spo2_icu,
 AVG(CASE WHEN di.label='Temperature Celsius' THEN ce.valuenum WHEN di.label='Temperature Fahrenheit' THEN (ce.valuenum-32)*5/9 END) temp_icu
FROM `{I}.chartevents` ce JOIN cohort c ON c.subject_id=ce.subject_id JOIN `{I}.d_items` di ON di.itemid=ce.itemid
WHERE ce.valuenum IS NOT NULL AND ce.hadm_id IS NOT NULL GROUP BY ce.hadm_id""").to_dataframe()
tri = client.query(f"""{CO}
SELECT t.stay_id, t.heartrate hr_ed, t.o2sat spo2_ed, t.resprate vital_resprate, t.temperature temp_f, t.sbp vital_sbp, t.dbp vital_dbp, t.chiefcomplaint
FROM `{ED}.triage` t JOIN cohort c ON c.subject_id=t.subject_id""").to_dataframe()
tri['temp_ed'] = (tri['temp_f']-32)*5/9

f = (link.merge(demo,on='subject_id',how='left')
         .merge(labs,on='hadm_id',how='left').merge(vicu,on='hadm_id',how='left')
         .merge(tri[['stay_id','hr_ed','spo2_ed','temp_ed','vital_resprate','vital_sbp','vital_dbp','chiefcomplaint']],on='stay_id',how='left'))
f['vital_heart_rate']=f['hr_icu'].fillna(f['hr_ed']); f['vital_spo2']=f['spo2_icu'].fillna(f['spo2_ed']); f['vital_temperature']=f['temp_icu'].fillna(f['temp_ed'])
f['ed_cough']=f['chiefcomplaint'].fillna('').str.contains('cough',case=False).astype(int)
assert (f.subject_id.isin(COH)).all()

micro = client.query(f"{CO} SELECT m.subject_id, m.spec_type_desc specimen, m.org_name FROM `{H}.microbiologyevents` m JOIN cohort c ON c.subject_id=m.subject_id").to_dataframe()
def onehot(df,col,k,pfx):
    s=df.dropna(subset=[col])
    if len(s)==0: return pd.DataFrame({'subject_id':[]})
    top=s[col].value_counts().head(k).index; s=s[s[col].isin(top)]
    oh=(pd.crosstab(s.subject_id,s[col])>0).astype(int); oh.columns=[pfx+str(c).strip().replace(' ','_').replace('/','_') for c in oh.columns]
    return oh.reset_index()
org_oh=onehot(micro,'org_name',20,'org_'); spec_oh=onehot(micro,'specimen',15,'spec_')
has_pos=micro.dropna(subset=['org_name']).groupby('subject_id').size().rename('n_positive_cultures').reset_index()
flags = client.query(f"""{CO}
SELECT d.subject_id,
 MAX(CASE WHEN (d.icd_version=10 AND (d.icd_code LIKE 'F17%' OR d.icd_code IN('Z720','Z87891'))) OR (d.icd_version=9 AND d.icd_code IN('3051','V1582')) THEN 1 ELSE 0 END) smoke,
 MAX(CASE WHEN (d.icd_version=10 AND d.icd_code LIKE 'R05%') OR (d.icd_version=9 AND d.icd_code='7862') THEN 1 ELSE 0 END) cough,
 MAX(CASE WHEN (d.icd_version=10 AND (d.icd_code LIKE 'E08%' OR d.icd_code LIKE 'E09%' OR d.icd_code LIKE 'E10%' OR d.icd_code LIKE 'E11%' OR d.icd_code LIKE 'E13%'))
            OR (d.icd_version=9 AND d.icd_code LIKE '250%') THEN 1 ELSE 0 END) diabetes
FROM `{H}.diagnoses_icd` d JOIN cohort c ON c.subject_id=d.subject_id GROUP BY d.subject_id""").to_dataframe()
for t_ in (org_oh,spec_oh,has_pos,flags): f=f.merge(t_,on='subject_id',how='left')

cols=['dicom_id','subject_id','study_id','hadm_id','stay_id','age','gender',
 'lab_hemoglobin','lab_wbc','lab_rbc','lab_platelet','lab_creatinine','lab_sodium','lab_potassium','lab_glucose','lab_bun','lab_chloride',
 'vital_heart_rate','vital_spo2','vital_temperature','vital_resprate','vital_sbp','vital_dbp','chiefcomplaint','ed_cough',
 'smoke','cough','diabetes','n_positive_cultures']+[c for c in f.columns if c.startswith('org_') or c.startswith('spec_')]
ehr = f[[c for c in cols if c in f.columns]].copy()
numc=[c for c in ehr.columns if c.startswith('lab_') or c.startswith('vital_') or c.startswith('org_') or c.startswith('spec_') or c in ('age','n_positive_cultures','smoke','cough','ed_cough','diabetes')]
for c in numc: ehr[c]=pd.to_numeric(ehr[c], errors='coerce')
ehr.to_parquet(os.path.join(OUT_DIR, f'ehr_{MODE}.parquet'), index=False)
print(f"[{MODE}] EHR:", ehr.shape, "| vital cov:", round(ehr.vital_heart_rate.notna().mean(),3),
      "| lab cov:", round(ehr.lab_hemoglobin.notna().mean(),3),
      "| diabetes:", int(ehr.drop_duplicates('subject_id').diabetes.sum()))


[encounter] CXR 231684 | hadm 178646 | ED 89109 | none 34738
[encounter] EHR: (231684, 64) | vital cov: 0.684 | lab cov: 0.72 | diabetes: 8516


In [ ]:
import pandas as pd, os
enc = pd.read_parquet(os.path.join(OUT_DIR,'ehr_encounter.parquet'))
fa  = pd.read_parquet(os.path.join(OUT_DIR,'ehr_firstadm.parquet'))

fill_cols = [c for c in enc.columns if c.startswith('lab_') or c.startswith('vital_')]
fa_base = fa.groupby('subject_id')[fill_cols].first()      # baseline 1 giá trị/bệnh nhân

hyb = enc.copy()
for c in fill_cols:
    hyb[c] = hyb[c].fillna(hyb['subject_id'].map(fa_base[c]))   # encounter -> NaN thì lấy baseline

hyb.to_parquet(os.path.join(OUT_DIR,'ehr_hybrid.parquet'), index=False)
cov = {c: round(hyb[c].notna().mean(),3) for c in
       ['vital_heart_rate','vital_spo2','vital_temperature','lab_hemoglobin','lab_wbc']}
print("hybrid coverage:", cov, "| shape:", hyb.shape)


hybrid coverage: {'vital_heart_rate': np.float64(0.734), 'vital_spo2': np.float64(0.732), 'vital_temperature': np.float64(0.729), 'lab_hemoglobin': np.float64(0.959), 'lab_wbc': np.float64(0.957)} | shape: (231684, 64)


## 11. Dataset fusion cuối: [EHR features] + [768 feature ảnh, mean CLS qua MỌI ảnh/bệnh nhân]\n\nLưu ý: bản gốc gộp embedding qua tất cả study của bệnh nhân, trong khi EHR lấy tại lần nhập viện đầu — có lệch thời gian; nếu cần khớp thời gian dùng bản first-obs.

In [ ]:
# import numpy as np, pandas as pd, os

# TOPK_ORG, TOPK_SPEC = 20, 15
# def onehot_topk(df, col, k, prefix):
#     s = df.dropna(subset=[col])
#     if len(s)==0: return pd.DataFrame({'subject_id':[]})
#     top = s[col].value_counts().head(k).index
#     s = s[s[col].isin(top)]
#     oh = (pd.crosstab(s['subject_id'], s[col])>0).astype(int)
#     oh.columns = [prefix+str(c).strip().replace(' ','_').replace('/','_') for c in oh.columns]
#     return oh.reset_index()

# org_oh  = onehot_topk(micro, 'org_name', TOPK_ORG,  'org_')
# spec_oh = onehot_topk(micro, 'specimen', TOPK_SPEC, 'spec_')
# has_pos = (micro.dropna(subset=['org_name']).groupby('subject_id').size()
#            .rename('n_positive_cultures').reset_index())

# # 768: mean CLS qua MỌI ảnh của bệnh nhân
# emb = np.load(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'))
# idx = pd.read_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'))
# edf = pd.DataFrame(emb, columns=[f'img_{i}' for i in range(emb.shape[1])])
# edf['subject_id'] = idx['subject_id'].values
# emb_pat = edf.groupby('subject_id', as_index=False).mean()

# # nhãn (max) + split + số ảnh gộp / bệnh nhân
# cm = pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))
# lab_cols = ['y_'+c for c in CHEXPERT_LABELS]
# agg = {c:'max' for c in lab_cols}; agg['split']='first'
# pat = cm.groupby('subject_id').agg(agg).reset_index()
# pat = pat.merge(cm.groupby('subject_id').size().rename('n_images').reset_index(), on='subject_id')

# fusion = (pat
#     .merge(anchor,  on='subject_id', how='left')
#     .merge(demo,    on='subject_id', how='left')
#     .merge(labs,    on='subject_id', how='left')
#     .merge(vit,     on='subject_id', how='left')
#     .merge(org_oh,  on='subject_id', how='left')
#     .merge(spec_oh, on='subject_id', how='left')
#     .merge(has_pos, on='subject_id', how='left')
#     .merge(flags,   on='subject_id', how='left')
#     .merge(emb_pat, on='subject_id', how='left'))

# zero_cols = [c for c in fusion.columns if c.startswith('org_') or c.startswith('spec_')] + ['n_positive_cultures','smoke','cough']
# fusion[zero_cols] = fusion[zero_cols].fillna(0)

# fusion.to_parquet(os.path.join(OUT_DIR,'fusion_dataset.parquet'), index=False)
# fusion.to_csv(os.path.join(OUT_DIR,'fusion_dataset.csv'), index=False)
# img_cols = [c for c in fusion.columns if c.startswith('img_')]
# ehr_cols = [c for c in fusion.columns if c not in img_cols]
# print("fusion_dataset:", fusion.shape, "| bệnh nhân:", fusion.subject_id.nunique())
# print("Số feature ảnh:", len(img_cols), "| số cột EHR/meta:", len(ehr_cols))
# print("Cột EHR/meta:", ehr_cols)

In [ ]:
import numpy as np, pandas as pd, os
EHR_SOURCE = 'hybrid'   # 'encounter' | 'firstadm' | 'hybrid'

emb = np.load(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'))
idx = pd.read_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'))
imgdf = pd.DataFrame(emb, columns=[f'img_{i}' for i in range(emb.shape[1])]); imgdf.insert(0,'dicom_id',idx['dicom_id'].values)

cm = pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))
lab_cols = ['y_'+c for c in CHEXPERT_LABELS]
extra    = [c for c in cm.columns if c=='disease_group' or c.startswith('dx_')]
meta = cm[['dicom_id','subject_id','study_id','split','view']+lab_cols+extra].drop_duplicates('dicom_id')

ehr = pd.read_parquet(os.path.join(OUT_DIR, f'ehr_{EHR_SOURCE}.parquet')).drop(columns=['subject_id','study_id'], errors='ignore')

fusion = meta.merge(imgdf, on='dicom_id', how='inner').merge(ehr, on='dicom_id', how='left')
zc = [c for c in fusion.columns if c.startswith('org_') or c.startswith('spec_')] + ['n_positive_cultures','smoke','cough','ed_cough','diabetes']
fusion[[c for c in zc if c in fusion.columns]] = fusion[[c for c in zc if c in fusion.columns]].fillna(0)

out = os.path.join(OUT_DIR, f'fusion_dataset_{EHR_SOURCE}')
fusion.to_parquet(out+'.parquet', index=False)
fusion[[c for c in fusion.columns if not c.startswith('img_')]].to_csv(out+'.csv', index=False)
img = [c for c in fusion.columns if c.startswith('img_')]
print(f"[{EHR_SOURCE}] fusion:", fusion.shape, "| ảnh:", fusion.dicom_id.nunique(),
      "| feature ảnh:", len(img), "| cột EHR/meta/nhãn:", fusion.shape[1]-len(img),
      "| diabetes:", int(fusion.drop_duplicates('subject_id').diabetes.sum()) if 'diabetes' in fusion.columns else 'N/A')


[hybrid] fusion: (231684, 857) | ảnh: 231684 | feature ảnh: 768 | cột EHR/meta/nhãn: 89 | diabetes: 8516


In [ ]:
import pandas as pd, numpy as np, os
from google.cloud import bigquery
client = bigquery.Client(project=BILLING_PROJECT)
H = "physionet-data.mimiciv_3_1_hosp"
BASE_SRC = 'hybrid'    # nền để thay micro: 'hybrid' | 'encounter' | 'firstadm'

# 1) toàn bộ microbiology của cohort
micro = client.query(f"""
WITH cohort AS (SELECT DISTINCT subject_id FROM ({TARGET_ADM}))
SELECT m.subject_id, m.spec_type_desc AS specimen, m.org_name
FROM `{H}.microbiologyevents` m JOIN cohort c ON c.subject_id=m.subject_id
""").to_dataframe()

# 2) one-hot TẤT CẢ (không top-K)
def onehot_full(df, col, pfx):
    s = df.dropna(subset=[col])
    if len(s)==0: return pd.DataFrame({'subject_id':[]})
    oh = (pd.crosstab(s['subject_id'], s[col])>0).astype('int8')
    oh.columns = [pfx+''.join(ch if ch.isalnum() else '_' for ch in str(c).strip()) for c in oh.columns]
    return oh.reset_index()
org_full  = onehot_full(micro, 'org_name', 'org_')
spec_full = onehot_full(micro, 'specimen', 'spec_')
print("Số cột org:", org_full.shape[1]-1, "| spec:", spec_full.shape[1]-1)

# 3) nạp fusion nền, bỏ org_/spec_ cũ (top-K), gắn bản full theo subject_id
fus = pd.read_parquet(os.path.join(OUT_DIR, f'fusion_dataset_{BASE_SRC}.parquet'))
fus = fus.drop(columns=[c for c in fus.columns if c.startswith('org_') or c.startswith('spec_')])
fus = fus.merge(org_full, on='subject_id', how='left').merge(spec_full, on='subject_id', how='left')
oh = [c for c in fus.columns if c.startswith('org_') or c.startswith('spec_')]
fus[oh] = fus[oh].fillna(0).astype('int8')

# 4) lưu bản mới
out = os.path.join(OUT_DIR, f'fusion_dataset_{BASE_SRC}_fullmicro')
fus.to_parquet(out+'.parquet', index=False)
fus[[c for c in fus.columns if not c.startswith('img_')]].to_csv(out+'.csv', index=False)
print("fusion fullmicro:", fus.shape, "| tổng cột org+spec:", len(oh),
      "->", f"fusion_dataset_{BASE_SRC}_fullmicro.parquet/.csv")


Số cột org: 617 | spec: 95
fusion fullmicro: (231684, 1534) | tổng cột org+spec: 712 -> fusion_dataset_hybrid_fullmicro.parquet/.csv


In [ ]:
import pandas as pd, os
df = pd.read_parquet(os.path.join(OUT_DIR,'fusion_dataset_hybrid.parquet'))   # <-- nạp lại
print(df.shape, "| img cols:", len([c for c in df.columns if c.startswith('img_')]))  # (210989, 860) | 768
df.to_csv(os.path.join(OUT_DIR,'fusion_dataset_hybrid_full.csv'), index=False)  # ~1.5GB, chậm


(231684, 857) | img cols: 768


In [ ]:
import pandas as pd, os
df = pd.read_parquet(os.path.join(OUT_DIR,'fusion_dataset_hybrid_fullmicro.parquet'))   # <-- nạp lại
print(df.shape, "| img cols:", len([c for c in df.columns if c.startswith('img_')]))  # (210989, 860) | 768
df.to_csv(os.path.join(OUT_DIR,'fusion_dataset_hybrid_fullmicro.csv'), index=False)  # ~1.5GB, chậm


(231684, 1534) | img cols: 768


In [ ]:
print(OUT_DIR)

/content/drive/MyDrive/MIMIC_CXR/planB_out


In [ ]:
import pandas as pd, os
df = pd.read_csv(os.path.join(OUT_DIR, 'fusion_dataset_hybrid_fullmicro.csv'))
df.head(10)


,dicom_id,subject_id,study_id,split,view,y_Atelectasis,y_Cardiomegaly,y_Consolidation,y_Edema,y_Enlarged Cardiomediastinum,...,spec_URINE,spec_URINE_KIDNEY,spec_URINE_PROSTATIC_MASSAGE,spec_URINE_SUPRAPUBIC_ASPIRATE,spec_VARICELLA_ZOSTER_CULTURE,spec_VIRAL_CULTURE,spec_VIRAL_CULTURE__R_O_CYTOMEGALOVIRUS,spec_VIRAL_CULTURE_R_O_HERPES_SIMPLEX_VIRUS,spec_WORM,spec_XXX
0,096052b7-d256dc40-453a102b-fa7d01c6-1b22c6b4,10000764,57375967,train,frontal,0.0,0.0,1.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
1,b79e55c3-735ce5ac-64412506-cdc9ea79-f1af521f,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
2,dcfeeac4-1597e318-d0e6736a-8b2c2238-47ac3f1b,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
3,d0b71acc-b5a62046-bbb5f6b8-7b173b85-65cdf738,10000935,50578979,train,frontal,0.0,0.0,0.0,1.0,0.0,...,1,0,0,0,0,0,0,0,0,0
4,3be619d1-506a66cf-ff1ab8a1-2efb77bb-fe7d59fc,10000935,51178377,train,lateral,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
5,9b314ad7-fbcb0422-6db62dfc-732858d0-a5527d8b,10000935,51178377,train,frontal,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
6,6a5c3985-7764bdd0-ec5a6a74-af78bcaa-4ca33ec3,10000935,55697293,train,lateral,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
7,c50494f1-90e2bff5-e9189550-1a4562fd-6ab5204c,10000935,55697293,train,frontal,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
8,8e3f2822-0c1d4b71-2a265bbf-5b96e531-ccf5fa30,10000935,56164612,train,frontal,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
9,ad13cf84-62c34a01-a01b9e87-2581a359-83bbc046,10000935,56164612,train,lateral,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0


In [ ]:
df.head(10)

,dicom_id,subject_id,study_id,split,view,y_Atelectasis,y_Cardiomegaly,y_Consolidation,y_Edema,y_Enlarged Cardiomediastinum,...,spec_URINE,spec_URINE_KIDNEY,spec_URINE_PROSTATIC_MASSAGE,spec_URINE_SUPRAPUBIC_ASPIRATE,spec_VARICELLA_ZOSTER_CULTURE,spec_VIRAL_CULTURE,spec_VIRAL_CULTURE__R_O_CYTOMEGALOVIRUS,spec_VIRAL_CULTURE_R_O_HERPES_SIMPLEX_VIRUS,spec_WORM,spec_XXX
0,096052b7-d256dc40-453a102b-fa7d01c6-1b22c6b4,10000764,57375967,train,frontal,0.0,0.0,1.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
1,b79e55c3-735ce5ac-64412506-cdc9ea79-f1af521f,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
2,dcfeeac4-1597e318-d0e6736a-8b2c2238-47ac3f1b,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
3,d0b71acc-b5a62046-bbb5f6b8-7b173b85-65cdf738,10000935,50578979,train,frontal,0.0,0.0,0.0,1.0,0.0,...,1,0,0,0,0,0,0,0,0,0
4,3be619d1-506a66cf-ff1ab8a1-2efb77bb-fe7d59fc,10000935,51178377,train,lateral,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
5,9b314ad7-fbcb0422-6db62dfc-732858d0-a5527d8b,10000935,51178377,train,frontal,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
6,6a5c3985-7764bdd0-ec5a6a74-af78bcaa-4ca33ec3,10000935,55697293,train,lateral,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
7,c50494f1-90e2bff5-e9189550-1a4562fd-6ab5204c,10000935,55697293,train,frontal,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
8,8e3f2822-0c1d4b71-2a265bbf-5b96e531-ccf5fa30,10000935,56164612,train,frontal,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
9,ad13cf84-62c34a01-a01b9e87-2581a359-83bbc046,10000935,56164612,train,lateral,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0


In [ ]:
import pandas as pd, os
df = pd.read_parquet(os.path.join(OUT_DIR,'fusion_dataset_hybrid.parquet'))   # <-- nạp lại
print(df.shape, "| img cols:", len([c for c in df.columns if c.startswith('img_')]))  # (210989, 860) | 768
df.to_csv(os.path.join(OUT_DIR,'fusion_dataset_hybrid_full.csv'), index=False)  # ~1.5GB, chậm


(210989, 860) | img cols: 768


In [ ]:
df.head(10)

,dicom_id,subject_id,study_id,split,view,y_Atelectasis,y_Cardiomegaly,y_Consolidation,y_Edema,y_Enlarged Cardiomediastinum,...,spec_IMMUNOLOGY,spec_MRSA_SCREEN,spec_PERITONEAL_FLUID,spec_PLEURAL_FLUID,spec_SEROLOGY_BLOOD,spec_SPUTUM,spec_STOOL,spec_SWAB,spec_TISSUE,spec_URINE
0,096052b7-d256dc40-453a102b-fa7d01c6-1b22c6b4,10000764,57375967,train,frontal,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,b79e55c3-735ce5ac-64412506-cdc9ea79-f1af521f,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,dcfeeac4-1597e318-d0e6736a-8b2c2238-47ac3f1b,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,d0b71acc-b5a62046-bbb5f6b8-7b173b85-65cdf738,10000935,50578979,train,frontal,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,3be619d1-506a66cf-ff1ab8a1-2efb77bb-fe7d59fc,10000935,51178377,train,lateral,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
5,9b314ad7-fbcb0422-6db62dfc-732858d0-a5527d8b,10000935,51178377,train,frontal,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
6,6a5c3985-7764bdd0-ec5a6a74-af78bcaa-4ca33ec3,10000935,55697293,train,lateral,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7,c50494f1-90e2bff5-e9189550-1a4562fd-6ab5204c,10000935,55697293,train,frontal,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
8,8e3f2822-0c1d4b71-2a265bbf-5b96e531-ccf5fa30,10000935,56164612,train,frontal,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
9,ad13cf84-62c34a01-a01b9e87-2581a359-83bbc046,10000935,56164612,train,lateral,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
from google.cloud import bigquery
import pandas as pd, os, glob
client = bigquery.Client(project=BILLING_PROJECT)
TARGET_DIR = OUT_DIR   # planB_out

GROUPS = list(DISEASE_TITLE.keys())
dx_sel = ",\n  ".join(f"MAX(CASE WHEN {group_cond(g)} THEN 1 ELSE 0 END) AS dx_{g}" for g in GROUPS)
dxg = client.query(f"""
WITH cohort AS (SELECT DISTINCT subject_id FROM ({TARGET_ADM}))
SELECT d.subject_id, {dx_sel}
FROM `{H}.diagnoses_icd` d JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
JOIN cohort c USING (subject_id) GROUP BY d.subject_id
""").to_dataframe()
_cp = os.path.join(TARGET_DIR,'cohort_subjects.parquet')
if os.path.exists(_cp):
    dxg = dxg[dxg.subject_id.isin(set(pd.read_parquet(_cp).subject_id))].reset_index(drop=True)
dxcols=['dx_'+g for g in GROUPS]
dxg['disease_group']=dxg[dxcols].apply(lambda r:'|'.join(g for g,v in zip(GROUPS,r.values) if v==1),axis=1)
allc=dxcols+['disease_group']

# manifest
mc=os.path.join(TARGET_DIR,'manifest_cohort.parquet')
if os.path.exists(mc):
    m=pd.read_parquet(mc).drop(columns=[c for c in allc],errors='ignore').merge(dxg,on='subject_id',how='left'); m.to_parquet(mc,index=False)

# CHỈ ghi parquet, KHÔNG đụng .csv
for p in glob.glob(os.path.join(TARGET_DIR,'fusion_dataset_*.parquet')):
    f=pd.read_parquet(p); nimg=len([c for c in f.columns if c.startswith('img_')])
    f=f.drop(columns=[c for c in allc],errors='ignore').merge(dxg,on='subject_id',how='left')
    f.to_parquet(p,index=False)
    print(f"  +disease_group: {os.path.basename(p)} {f.shape} (img: {nimg})  [csv KHÔNG đụng]")
print("Xong. File .csv của bạn giữ nguyên.")


  +disease_group: fusion_dataset_hybrid.parquet (231684, 857) (img: 768)  [csv KHÔNG đụng]
  +disease_group: fusion_dataset_hybrid_fullmicro.parquet (231684, 1534) (img: 768)  [csv KHÔNG đụng]
Xong. File .csv của bạn giữ nguyên.


In [ ]:
import pandas as pd, os
df = pd.read_parquet(os.path.join(OUT_DIR,'fusion_dataset_hybrid_fullmicro.parquet'))   # <-- nạp lại
print(df.shape, "| img cols:", len([c for c in df.columns if c.startswith('img_')]))  # (210989, 860) | 768
df.to_csv(os.path.join(OUT_DIR,'fusion_dataset_hybrid_fullmicro.csv'), index=False)  # ~1.5GB, chậm


(231684, 1534) | img cols: 768


In [ ]:
import pandas as pd, os
df = pd.read_csv(os.path.join(OUT_DIR, 'fusion_dataset_hybrid_fullmicro.csv'))
df.head(10)


,dicom_id,subject_id,study_id,split,view,y_Atelectasis,y_Cardiomegaly,y_Consolidation,y_Edema,y_Enlarged Cardiomediastinum,...,spec_XXX,dx_pneumonia,dx_bronchitis,dx_tuberculosis,dx_arthritis,dx_fracture,dx_scoliosis,dx_lung_cancer,dx_lung_infection,disease_group
0,096052b7-d256dc40-453a102b-fa7d01c6-1b22c6b4,10000764,57375967,train,frontal,0.0,0.0,1.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,fracture
1,b79e55c3-735ce5ac-64412506-cdc9ea79-f1af521f,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,fracture
2,dcfeeac4-1597e318-d0e6736a-8b2c2238-47ac3f1b,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,fracture
3,d0b71acc-b5a62046-bbb5f6b8-7b173b85-65cdf738,10000935,50578979,train,frontal,0.0,0.0,0.0,1.0,0.0,...,0,0,0,0,1,0,0,1,0,arthritis|lung_cancer
4,3be619d1-506a66cf-ff1ab8a1-2efb77bb-fe7d59fc,10000935,51178377,train,lateral,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,1,0,arthritis|lung_cancer
5,9b314ad7-fbcb0422-6db62dfc-732858d0-a5527d8b,10000935,51178377,train,frontal,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,1,0,arthritis|lung_cancer
6,6a5c3985-7764bdd0-ec5a6a74-af78bcaa-4ca33ec3,10000935,55697293,train,lateral,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,1,0,arthritis|lung_cancer
7,c50494f1-90e2bff5-e9189550-1a4562fd-6ab5204c,10000935,55697293,train,frontal,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,1,0,arthritis|lung_cancer
8,8e3f2822-0c1d4b71-2a265bbf-5b96e531-ccf5fa30,10000935,56164612,train,frontal,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,1,0,arthritis|lung_cancer
9,ad13cf84-62c34a01-a01b9e87-2581a359-83bbc046,10000935,56164612,train,lateral,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,1,0,arthritis|lung_cancer


In [ ]:
import pandas as pd, os
df = pd.read_parquet(os.path.join(OUT_DIR,'fusion_dataset_hybrid.parquet'))   # <-- nạp lại
print(df.shape, "| img cols:", len([c for c in df.columns if c.startswith('img_')]))  # (210989, 860) | 768
df.to_csv(os.path.join(OUT_DIR,'fusion_dataset_hybrid_full.csv'), index=False)  # ~1.5GB, chậm


In [ ]:
import pandas as pd, os
df = pd.read_csv(os.path.join(OUT_DIR, 'fusion_dataset_hybrid_full.csv'))
df.head(10)


,dicom_id,subject_id,study_id,split,view,y_Atelectasis,y_Cardiomegaly,y_Consolidation,y_Edema,y_Enlarged Cardiomediastinum,...,spec_IMMUNOLOGY,spec_MRSA_SCREEN,spec_PERITONEAL_FLUID,spec_PLEURAL_FLUID,spec_SEROLOGY_BLOOD,spec_SPUTUM,spec_STOOL,spec_SWAB,spec_TISSUE,spec_URINE
0,096052b7-d256dc40-453a102b-fa7d01c6-1b22c6b4,10000764,57375967,train,frontal,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,b79e55c3-735ce5ac-64412506-cdc9ea79-f1af521f,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,dcfeeac4-1597e318-d0e6736a-8b2c2238-47ac3f1b,10000764,57375967,train,lateral,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,d0b71acc-b5a62046-bbb5f6b8-7b173b85-65cdf738,10000935,50578979,train,frontal,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,3be619d1-506a66cf-ff1ab8a1-2efb77bb-fe7d59fc,10000935,51178377,train,lateral,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
5,9b314ad7-fbcb0422-6db62dfc-732858d0-a5527d8b,10000935,51178377,train,frontal,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
6,6a5c3985-7764bdd0-ec5a6a74-af78bcaa-4ca33ec3,10000935,55697293,train,lateral,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7,c50494f1-90e2bff5-e9189550-1a4562fd-6ab5204c,10000935,55697293,train,frontal,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
8,8e3f2822-0c1d4b71-2a265bbf-5b96e531-ccf5fa30,10000935,56164612,train,frontal,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
9,ad13cf84-62c34a01-a01b9e87-2581a359-83bbc046,10000935,56164612,train,lateral,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
from google.cloud import bigquery
import pandas as pd, os, glob
client = bigquery.Client(project=BILLING_PROJECT)

TARGET_DIR = OUT_DIR          # planB_out
CSV_MODE   = 'full'          # 'light' = CSV không kèm 768 img (nhẹ) ; 'full' = kèm img (~GB, chậm)

GROUPS = list(DISEASE_TITLE.keys())
dx_sel = ",\n  ".join(f"MAX(CASE WHEN {group_cond(g)} THEN 1 ELSE 0 END) AS dx_{g}" for g in GROUPS)
dxg = client.query(f"""
WITH cohort AS (SELECT DISTINCT subject_id FROM ({TARGET_ADM}))
SELECT d.subject_id, {dx_sel}
FROM `{H}.diagnoses_icd` d JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
JOIN cohort c USING (subject_id) GROUP BY d.subject_id
""").to_dataframe()
_cp = os.path.join(TARGET_DIR,'cohort_subjects.parquet')
if os.path.exists(_cp):
    dxg = dxg[dxg.subject_id.isin(set(pd.read_parquet(_cp).subject_id))].reset_index(drop=True)
dxcols=['dx_'+g for g in GROUPS]
dxg['disease_group']=dxg[dxcols].apply(lambda r:'|'.join(g for g,v in zip(GROUPS,r.values) if v==1),axis=1)
allc=dxcols+['disease_group']

mc=os.path.join(TARGET_DIR,'manifest_cohort.parquet')
if os.path.exists(mc):
    m=pd.read_parquet(mc).drop(columns=[c for c in allc],errors='ignore').merge(dxg,on='subject_id',how='left'); m.to_parquet(mc,index=False)

for p in glob.glob(os.path.join(TARGET_DIR,'fusion_dataset_*.parquet')):
    f=pd.read_parquet(p); nimg=len([c for c in f.columns if c.startswith('img_')])
    f=f.drop(columns=[c for c in allc],errors='ignore').merge(dxg,on='subject_id',how='left')
    f.to_parquet(p,index=False)
    cols = list(f.columns) if CSV_MODE=='full' else [c for c in f.columns if not c.startswith('img_')]
    f[cols].to_csv(p.replace('.parquet','.csv'), index=False)
    print(f"  +disease_group: {os.path.basename(p)} parquet{f.shape} | csv[{CSV_MODE}] {len(cols)} cột (img:{nimg})")
print("Xong.")


  +disease_group: fusion_dataset_hybrid.parquet parquet(210989, 860) | csv[full] 860 cột (img:768)
  +disease_group: fusion_dataset_hybrid_fullmicro.parquet parquet(210989, 1527) | csv[full] 1527 cột (img:768)
Xong.


In [ ]:
import pandas as pd, os
p = os.path.join(OUT_DIR,'fusion_dataset_hybrid_full.csv')

# đọc CSV, bỏ qua 768 cột ảnh cho nhanh (vẫn có disease_group + EHR + nhãn)
df = pd.read_csv(p)
print("shape (không img):", df.shape)
print("cột disease:", [c for c in df.columns if c=='disease_group' or c.startswith('dx_')])
print(df.drop_duplicates('subject_id')['disease_group'].value_counts().head(15))


shape (không img): (231684, 857)
cột disease: ['dx_pneumonia', 'dx_bronchitis', 'dx_tuberculosis', 'dx_arthritis', 'dx_fracture', 'dx_scoliosis', 'dx_lung_cancer', 'dx_lung_infection', 'disease_group']
disease_group
pneumonia                       5281
fracture                        4234
arthritis                       3754
pneumonia|fracture              1574
pneumonia|arthritis             1321
arthritis|fracture              1160
lung_cancer                     1151
pneumonia|arthritis|fracture     691
pneumonia|lung_cancer            622
lung_infection                   531
pneumonia|lung_infection         474
pneumonia|bronchitis             392
bronchitis                       364
tuberculosis                     288
fracture|lung_cancer             257
Name: count, dtype: int64


In [ ]:
import pandas as pd, os
p = os.path.join(OUT_DIR,'fusion_dataset_hybrid_fullmicro.csv')

# đọc CSV, bỏ qua 768 cột ảnh cho nhanh (vẫn có disease_group + EHR + nhãn)
df = pd.read_csv(p)
print("shape (không img):", df.shape)
print("cột disease:", [c for c in df.columns if c=='disease_group' or c.startswith('dx_')])
print(df.drop_duplicates('subject_id')['disease_group'].value_counts().head(15))


shape (không img): (231684, 1534)
cột disease: ['dx_pneumonia', 'dx_bronchitis', 'dx_tuberculosis', 'dx_arthritis', 'dx_fracture', 'dx_scoliosis', 'dx_lung_cancer', 'dx_lung_infection', 'disease_group']
disease_group
pneumonia                       5281
fracture                        4234
arthritis                       3754
pneumonia|fracture              1574
pneumonia|arthritis             1321
arthritis|fracture              1160
lung_cancer                     1151
pneumonia|arthritis|fracture     691
pneumonia|lung_cancer            622
lung_infection                   531
pneumonia|lung_infection         474
pneumonia|bronchitis             392
bronchitis                       364
tuberculosis                     288
fracture|lung_cancer             257
Name: count, dtype: int64


In [ ]:
from google.cloud import bigquery
import pandas as pd, os, glob
client = bigquery.Client(project=BILLING_PROJECT)
H="physionet-data.mimiciv_3_1_hosp"; ED="physionet-data.mimiciv_ed"
CO = f"WITH cohort AS (SELECT DISTINCT subject_id FROM ({TARGET_ADM}))"

# --- study_dt: lúc chụp CXR (từ metadata) ---
meta = pd.read_csv(os.path.join(BASE,'mimic-cxr-2.0.0-metadata.csv.gz'),
                   usecols=['dicom_id','StudyDate','StudyTime'])
_d=meta.StudyDate.astype('int64').astype(str).str.zfill(8)
_t=meta.StudyTime.fillna(0).astype(float).astype('int64').astype(str).str.zfill(6).str.slice(0,6)
meta['study_dt']=pd.to_datetime(_d+_t, format='%Y%m%d%H%M%S', errors='coerce')
sdt = meta[['dicom_id','study_dt']].drop_duplicates('dicom_id'); sdt['dicom_id']=sdt.dicom_id.astype(str)

# --- admittime (nội trú) + ed_intime (ED) theo cohort ---
adm = client.query(f"{CO} SELECT a.hadm_id, a.admittime FROM `{H}.admissions` a JOIN cohort c ON c.subject_id=a.subject_id").to_dataframe()
adm['admittime']=pd.to_datetime(adm.admittime); adm['hadm_id']=adm.hadm_id.astype('Int64'); adm=adm.drop_duplicates('hadm_id')
edi = client.query(f"{CO} SELECT e.stay_id, e.intime AS ed_intime FROM `{ED}.edstays` e JOIN cohort c ON c.subject_id=e.subject_id").to_dataframe()
edi['ed_intime']=pd.to_datetime(edi.ed_intime); edi['stay_id']=edi.stay_id.astype('Int64'); edi=edi.drop_duplicates('stay_id')

TIMECOLS=['study_dt','admittime','ed_intime','encounter_time']
def add_time(df):
    df = df.drop(columns=TIMECOLS, errors='ignore'); df['dicom_id']=df.dicom_id.astype(str)
    df = df.merge(sdt, on='dicom_id', how='left')
    if 'hadm_id' in df.columns:
        df['hadm_id']=pd.to_numeric(df.hadm_id,errors='coerce').astype('Int64'); df=df.merge(adm,on='hadm_id',how='left')
    if 'stay_id' in df.columns:
        df['stay_id']=pd.to_numeric(df.stay_id,errors='coerce').astype('Int64'); df=df.merge(edi,on='stay_id',how='left')
    if 'admittime' not in df.columns: df['admittime']=pd.NaT
    if 'ed_intime' not in df.columns: df['ed_intime']=pd.NaT
    df['encounter_time']=df['admittime'].fillna(df['ed_intime'])
    return df

# 1) PARQUET — nguồn chuẩn, giữ đủ 768 ảnh
for p in glob.glob(os.path.join(OUT_DIR,'fusion_dataset_*.parquet')):
    df=add_time(pd.read_parquet(p)); df.to_parquet(p,index=False)
    print("parquet:",os.path.basename(p),"| img:",sum(c.startswith('img_') for c in df.columns),
          "| study_dt:",round(df.study_dt.notna().mean(),2),"| enc:",round(df.encounter_time.notna().mean(),2))

# 2) CSV — giữ nguyên dạng: full nếu vốn có ảnh, light nếu không
for cp in glob.glob(os.path.join(OUT_DIR,'fusion_dataset_*.csv')):
    had_img = any(c.startswith('img_') for c in pd.read_csv(cp,nrows=0).columns)
    df=add_time(pd.read_csv(cp))
    if not had_img: df=df[[c for c in df.columns if not c.startswith('img_')]]
    df.to_csv(cp,index=False)
    print("csv:",os.path.basename(cp),"| img giữ:",had_img,"|",df.shape)


parquet: fusion_dataset_hybrid.parquet | img: 768 | study_dt: 1.0 | enc: 0.85
parquet: fusion_dataset_hybrid_fullmicro.parquet | img: 768 | study_dt: 1.0 | enc: 0.85
csv: fusion_dataset_hybrid.csv | img giữ: False | (231684, 93)
csv: fusion_dataset_hybrid_full.csv | img giữ: True | (231684, 861)
csv: fusion_dataset_hybrid_fullmicro.csv | img giữ: True | (231684, 1538)
